# exp14: benefit router cascade (Colab GPU)

Runtime → Change runtime type → **T4 GPU**, then *Run all*. A1 is read from Drive (`MyDrive/qualiusp2026/exp02_results/full/model`); MuSiQue is downloaded from the official release; both are hash-checked.
Models and results are written to Drive (`MyDrive/qualiusp2026/exp12_<commit>`). Training checkpoints every 500 steps, so after a
disconnect just *Run all* again: finished models and evaluation configs are skipped. Code comes from the public repo (PROTOCOL.md). Models: exp12's frozen GPU models on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, subprocess
PIN = '11296db06843c8ad47aa67a8c5dfb60fdd04b5dc'  # registration commit (PROTOCOL: pinned code, fresh output folder)
OUT = f'/content/drive/MyDrive/qualiusp2026/exp14_{PIN[:7]}'
E13 = '/content/drive/MyDrive/qualiusp2026/exp13_d071b74/results'  # exp13 rankings with the same frozen models
M12 = '/content/drive/MyDrive/qualiusp2026/exp12_975371a/models'  # frozen exp12 models
A1 = '/content/drive/MyDrive/qualiusp2026/exp02_results/full/model'  # exp02's A1 (sha256 checked below)
pathlib.Path(OUT, 'models').mkdir(parents=True, exist_ok=True); pathlib.Path(OUT, 'results').mkdir(exist_ok=True)
if not os.path.exists('/content/qualiusp2026'):
    !git clone -q https://github.com/arnml/qualiusp2026.git /content/qualiusp2026
!cd /content/qualiusp2026 && git fetch -q && git checkout -q {PIN} && git log --oneline -1
assert subprocess.check_output(['git', '-C', '/content/qualiusp2026', 'rev-parse', 'HEAD']).decode().strip() == PIN
# Inputs (deviation 1): A1 from exp02's Drive folder; MuSiQue from the official release (StonyBrookNLP/musique).
# Both are checked against sha256 hashes of the local files used in exp08-exp11.
import hashlib, json
def sha(path): return hashlib.sha256(open(path, 'rb').read()).hexdigest()
assert sha(f'{A1}/model.safetensors') == '605c986eb9a9a82b10ac43c43c77b2a0f976c04c7ed40f21cc66f2a5b19d950d'
D = '/content/qualiusp2026/experiments/exp08_musique/data'
if os.path.islink(D): os.unlink(D)  # left by the first attempt (Drive inputs from another account)
if not os.path.exists(f'{D}/musique_ans_v1.0_dev.jsonl'):
    !pip -q install gdown && cd /content && gdown -q 1tGdADlNjWFaHLeZZGShh2IRcpO6Lv24h -O musique.zip && unzip -q -o musique.zip -d musique_raw
    os.makedirs(D, exist_ok=True)
    src = subprocess.check_output('find /content/musique_raw -name musique_ans_v1.0_dev.jsonl', shell=True).decode().split()[0].rsplit('/', 1)[0]
    !cp {src}/musique_ans_v1.0_dev.jsonl {D}/
    ids = json.load(open('/content/qualiusp2026/experiments/exp14_benefit_router/split_ids.json'))
    ids = set(ids['b_train']) | set(ids['b_tune']) | set(ids['b_eval']) | set(ids['router_train'])
    with open(f'{src}/musique_ans_v1.0_train.jsonl') as f, open(f'{D}/musique_ans_v1.0_train.jsonl', 'w') as g:
        for line in f:
            if json.loads(line)['id'] in ids: g.write(line)
assert sha(f'{D}/musique_ans_v1.0_dev.jsonl') == '15fa63794d18a94ce12411aca6e2327e65b6e83b0b1490efab3f1962e48abf3b', 'dev file differs'
assert sha(f'{D}/musique_ans_v1.0_train.jsonl') == '4093c17f2266fcf3fb0e3647aff18d892fc28edecea62f6f0609cd76d7f2c49b', 'train subset differs'
print('inputs verified')
os.environ.update(A1_DIR=A1, MODELS_DIR=M12, RESULTS_DIR=f'{OUT}/results', TOKENIZERS_PARALLELISM='false')
os.chdir('/content/qualiusp2026/experiments/exp14_benefit_router')
for t, h in (('m1_iter_s0', '2c91c40f1310ca02'), ('m1_ctrl', 'ed87244f462412f9')):
    assert sha(f'{M12}/{t}/model/model.safetensors')[:16] == h, t
import torch, transformers; print('torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install tiktoken
!git rev-parse HEAD > {OUT}/results/commit.txt

## Rank (frozen exp12 models; exp13's router_train and dev_fresh rankings reused), then the pre-registered analysis

In [ ]:
import shutil
for f in ('rank_router_train.jsonl', 'rank_dev_fresh.jsonl'):
    if not os.path.exists(f'{OUT}/results/{f}'): shutil.copy(f'{E13}/{f}', f'{OUT}/results/{f}')
for sp in ('b_train', 'b_tune', 'b_eval'):
    !python rank.py --split {sp} --only ctrl_oneshot iter_iter 2>&1 | grep -v -e Warning -e Loading -e 'Token indices'
!python rank.py --split b_eval --only iter_oneshot ctrl_iter 2>&1 | grep -v -e Warning -e Loading -e 'Token indices'
!python analyze.py